# Explainable Risk Intelligence — recruiter walkthrough

This notebook is a read-only tour of evidence generated by the package. The business question is whether a calibrated model can support transparent review of next-month credit-card default risk without pretending to be an automated lending policy. Run `make reproduce` before this notebook.

In [ ]:
from explainable_risk_intelligence.portfolio import final_metrics_table, load_portfolio_evidence
evidence = load_portfolio_evidence()
manifest = evidence['manifest']
print(f"Dataset rows: {manifest['rows']:,}; target: {manifest['target']}; default rate: {manifest['default_rate']:.3f}")

## Data boundaries and leakage contract
The official UCI data is split once with target stratification and seed 42. Training owns preprocessing and cross-validation; validation owns model, calibration, and illustrative threshold choices; final test evidence is loaded only after those choices are persisted.

In [ ]:
print(manifest['splits'])
print('Excluded from model:', manifest['excluded_from_model'])

## Model comparison and calibration

In [ ]:
print(evidence['model_comparison'][['model', 'average_precision', 'brier_score']].to_string(index=False))
print(evidence['calibration'][['model', 'average_precision', 'brier_score']].to_string(index=False))
print('Calibration improves both metrics, but the gain is intentionally described as marginal.')

## Locked final evaluation and illustrative 1:5 review policy

In [ ]:
print(final_metrics_table(evidence).to_string())

## Explainability
Permutation importance explains the calibrated prediction pipeline. SHAP explains the underlying uncalibrated HGB estimator in transformed feature space; local records retain both original and transformed values. Attribution is not causality.

In [ ]:
print(evidence['shap_global'].head(8).to_string(index=False))
for case in evidence['local_explanations']:
    print(case['case_label'], case['record_id'], round(case['predicted_default_probability'], 4))

## Fairness and error analysis
Sex and age are audit-only attributes. Group gaps and segment errors are descriptive benchmark observations, not proof of fairness, causation, or regulatory acceptability.

In [ ]:
print(evidence['fairness_groups'].to_string(index=False))
print(evidence['error_segments'].head(8).to_string(index=False))
for observation in evidence['error_analysis']['observations']:
    print('-', observation)

## Robustness and synthetic monitoring demonstrator

In [ ]:
print(evidence['robustness'])
print(evidence['drift_features'].to_string(index=False))
print(evidence['drift_summary'])

## Limits and responsible use
Historical 2005 data from one geography cannot establish current or cross-market performance. The random split does not test temporal stability. The threshold costs are illustrative, perturbations are scenarios rather than deployment evidence, and the model must not autonomously approve, decline, price, or limit credit.